# Confusion Matrix Heatmap

See **which classes get confused** after classification.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Train and predict

In [ ]:
gen = DummyDataGenerator(n_samples=300, n_features=8, n_classes=3)
X, y = gen.tensors()
model = SimpleMLP()
opt = torch.optim.Adam(model.parameters(), lr=0.02)
for _ in range(80):
    opt.zero_grad()
    F.cross_entropy(model(X), y).backward()
    opt.step()
with torch.no_grad():
    preds = model(X).argmax(1)

## 2. Build confusion matrix

In [ ]:
n_classes = 3
cm = torch.zeros(n_classes, n_classes, dtype=torch.int32)
for t, p in zip(y, preds):
    cm[t, p] += 1
print(cm)

## 3. Heatmap (counts)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm.numpy(), cmap='Blues')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(3):
    for j in range(3):
        ax.text(j, i, int(cm[i,j]), ha='center', va='center', color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.colorbar(im); ax.set_title('Confusion matrix (counts)')
plt.tight_layout(); plt.show()

## 4. Normalized confusion matrix

In [ ]:
cm_norm = cm.float() / cm.sum(dim=1, keepdim=True).clamp(min=1)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm.numpy(), cmap='Oranges', vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xlabel('predicted'); ax.set_ylabel('true')
plt.colorbar(im, label='recall per true class')
ax.set_title('Row-normalized confusion matrix')
plt.tight_layout(); plt.show()

## 5. Per-class accuracy bar chart

In [ ]:
diag = cm.diag().float()
row_sum = cm.sum(dim=1).float().clamp(min=1)
class_acc = (diag / row_sum).numpy()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(3), class_acc, color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='black')
ax.set_ylim(0, 1.05); ax.set_xlabel('class'); ax.set_ylabel('recall')
ax.set_title('Per-class recall from confusion matrix')
plt.tight_layout(); plt.show()